# Network Topology Metrics Analysis

This notebook loads a graph from a **GraphML** file and computes a comprehensive set of
network-science metrics that describe its structure, then visualizes the graph and its
degree distribution.

**Improvements made over the original notebook:**
1. Local (non-Colab) file path with existence checking and a clear error message.
2. All calculations wrapped in small, reusable functions.
3. Proper error handling for edge cases (disconnected graphs, single-node components,
   graphs where assortativity is undefined, etc.).
4. Fixed a naming bug: the original code stored `average_clustering()` in a variable
   called `transitivity`. Here both **global transitivity** (`nx.transitivity`) and
   **average clustering coefficient** (`nx.average_clustering`) are computed and
   reported separately, since they are different measures.
5. Removed a redundant/duplicate calculation of `isolated_nodes`.
6. Added graph visualization and a degree-distribution histogram.
7. Results are still displayed in a nicely formatted table via `tabulate`.

## Step 1 — Imports

In [ ]:
# --- Make the shared `techintel` package importable ---------------------------
# The WoS parsing / graph / metrics helpers used to be copy-pasted into each
# notebook. They now live in techintel/ at the repository root; this finds that
# root by walking up from the working directory.
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "techintel" / "__init__.py").exists():
            return candidate
    raise RuntimeError(
        "Could not find the repository root (a directory containing 'techintel/')."
    )


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import networkx as nx
from tabulate import tabulate

from techintel import graphs, metrics


## Step 2 — Load the graph

The path is configurable via the `GRAPHML_PATH` variable below. If the file is not found,
a clear `FileNotFoundError` is raised instead of an opaque NetworkX error.

In [ ]:
# Path to the GraphML file.
# Change this to point at your own file. Relative paths are resolved from the
# notebook's working directory (no hard-coded Colab '/content/' path).
GRAPHML_PATH = "SmallSampleGraph.graphml"

# `graphs.load_graph` raises FileNotFoundError naming the path if it is missing.
G = graphs.load_graph(GRAPHML_PATH)
print(f"Loaded graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")


## Step 3 — Basic structural counts (v, e, p, u) and alpha / beta / gamma indices

In [ ]:
# v/e/p/u counts and the alpha/beta/gamma indices come from techintel.metrics.
counts = metrics.basic_counts(G)
v, e, p, u = counts["v"], counts["e"], counts["p"], counts["u"]

indices = metrics.alpha_beta_gamma(v, e, u)
alpha, beta, gamma = indices["alpha"], indices["beta"], indices["gamma"]

print(counts)
print(indices)


## Step 4 — Degree-based metrics: average degree & network heterogeneity

In [ ]:
deg_stats = metrics.degree_stats(G)
avg_degree = deg_stats["avg_degree"]
heterogeneity = deg_stats["heterogeneity"]
degrees = deg_stats["degrees"]

print({"avg_degree": avg_degree, "heterogeneity": heterogeneity})


## Step 5 — Average shortest path length

Computed per connected component and then averaged. Components with fewer than 2
nodes are skipped since the average shortest path length is undefined for them
(NetworkX would otherwise raise `NetworkXPointlessConcept`).

In [ ]:
average_shortest_path_length = metrics.average_shortest_path_length_safe(G)

print({"average_shortest_path_length": average_shortest_path_length})


## Step 6 — Triangles, transitivity & clustering, isolated nodes, assortativity

In [ ]:
triangles = metrics.triangle_count(G)
isolated_nodes = metrics.isolated_node_count(G)

# Two distinct, correctly-named measures (the original notebook mislabeled
# average_clustering as "transitivity"):
global_transitivity = metrics.global_transitivity(G)    # fraction of transitive triples
average_clustering = metrics.average_clustering(G)      # mean of local clustering coefficients

assortativity = metrics.safe_assortativity(G)

print({
    "triangles": triangles,
    "isolated_nodes": isolated_nodes,
    "global_transitivity": global_transitivity,
    "average_clustering": average_clustering,
    "assortativity": assortativity,
})


## Step 7 — Results table

In [ ]:
results_table = [
    ["Nodes (v)", v],
    ["Edges (e)", e],
    ["Sub-graphs (p)", p],
    ["Cycles (u)", u],
    ["Alpha", alpha],
    ["Beta", beta],
    ["Gamma", gamma],
    ["Average Shortest Path Length", average_shortest_path_length],
    ["Triangles", triangles],
    ["Global Transitivity", global_transitivity],
    ["Average Clustering Coefficient", average_clustering],
    ["Isolated Nodes", isolated_nodes],
    ["Network Heterogeneity", heterogeneity],
    ["Assortativity", assortativity],
    ["Average Degree", avg_degree],
]

print(tabulate(results_table, headers=["Measure", "Value"]))

## Step 8 — Visualize the graph

In [ ]:
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(
    G,
    pos,
    with_labels=True,
    node_color="skyblue",
    edge_color="gray",
    node_size=500,
    font_size=8,
)
plt.title("Network Graph")
plt.tight_layout()
plt.show()

## Step 9 — Degree distribution histogram

In [ ]:
plt.figure(figsize=(8, 5))
if degrees:
    max_deg = max(degrees)
    plt.hist(degrees, bins=range(0, max_deg + 2), align="left", color="salmon", edgecolor="black")
plt.xlabel("Degree")
plt.ylabel("Number of Nodes")
plt.title("Degree Distribution")
plt.tight_layout()
plt.show()